# Chapter 40 — Build a Document Embedding Pipeline

Chapter 39, you built a semantic search engine over a list of sentences. Now we'll extend it to work with an actual PDF document.
1. What you'll build
PDF document

Extract text
Read the PDF's text content

Split into chunks
Create smaller text segments

Generate embeddings
Convert each chunk into a vector

Ready for semantic search

In [31]:
from sentence_transformers import SentenceTransformer

model_small = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Step 1 : Install the pdf library

In [1]:
%pip install pypdf sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# Step 2 - Load a PDF

In [3]:
from pypdf import PdfReader 

pdf_path = "Chapter 41 : Sample.pdf"

reader = PdfReader(pdf_path)

In [6]:
print("Number of pages:", len(reader.pages))

Number of pages: 30


# Step 3 : Extract text from every page

In [ ]:
pages = []
# reader.pages: gives us access to all the pages in the PDF.
# enumerate(): lets us loop through those pages while also keeping a counter.
# start=1: starts the counter at page 1 instead of page 0.

for page_number, page in enumerate(reader.pages, start = 1):
    text = page.extract_text()

    pages.append({"page" : page_number,
                  "text" : text})

print(len(pages))

30


In [11]:
print("First page preview")
pages[0]["text"][:1000]

First page preview


" 9\nUNIT 1  INTRODUCTION TO ARTIFICIAL     \n      INTELLIGENCE\nStructure\n1.1  Introduction\n1.2  Objectives\n1.3  Basics of Artificial Intelligence (AI)?\n1.4  Brief history of Artificial Intelligence \n1.5  Components of Intelligence\n1.6  Approaches to Artificial Intelligence\n1.7  Comparison between Artificial Intelligence (AI), Machine Learning  \n  (ML) and DeepLearning (DL).\n1.8  Application Areas of Artificial Intelligence Systems\n1.9  Intelligent Agents\n  1.9.1  Stimulus - Response Agents\n1.10 Summary\n1.11 Solutions/Answers\n1.12 Further Readings \n1.1 INTRODUCTION \nToday, artificial intelligence is used in a wide variety of applications, including \nengineering, technology, the military, opinion mining, sentiment analysis, \nand  many more. It is also used in more advanced domains, such as language \nprocessing and applications for aerospace.\nAI is everywhere in today's world, and people are gradually becoming accustomed \nto its presence. It is utilised in systems 

pages[0] gets the first dictionary. Python list indexing starts at 0.

["page"] retrieves its page number.

["text"] retrieves its extracted text.

[:1000] displays only the first 300 characters.

The key concept: you're not embedding the entire PDF as one giant piece of text. You're extracting the text page by page so you can split it into 
manageable chunks, generate embeddings for those chunks, and later retrieve the most relevant parts.

# Step 4: Clean the extracted text
PDF extraction can contain extra spaces and line breaks. Let's normalize the text before chunking.

In [15]:
import re 

def clean_text(text):
    if not text:
        return ""

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text )

    return text.strip()

In [17]:
for page in pages: 
    page["text"] = clean_text(page["text"])

In [19]:
pages[0]["text"[:500]]

"9 UNIT 1 INTRODUCTION TO ARTIFICIAL INTELLIGENCE Structure 1.1 Introduction 1.2 Objectives 1.3 Basics of Artificial Intelligence (AI)? 1.4 Brief history of Artificial Intelligence 1.5 Components of Intelligence 1.6 Approaches to Artificial Intelligence 1.7 Comparison between Artificial Intelligence (AI), Machine Learning (ML) and DeepLearning (DL). 1.8 Application Areas of Artificial Intelligence Systems 1.9 Intelligent Agents 1.9.1 Stimulus - Response Agents 1.10 Summary 1.11 Solutions/Answers 1.12 Further Readings 1.1 INTRODUCTION Today, artificial intelligence is used in a wide variety of applications, including engineering, technology, the military, opinion mining, sentiment analysis, and many more. It is also used in more advanced domains, such as language processing and applications for aerospace. AI is everywhere in today's world, and people are gradually becoming accustomed to its presence. It is utilised in systems that recognise both voices and faces. In addition to this, it

Now we don't have /n or any white spaces 

# Step 5 : Split the text into chunks 

An embedding model has practical input-length limits. Instead of embedding an entire document as one vector, we divide the text into smaller pieces.

In [24]:
def chunk_text(text, chunk_size = 800, overlap = 150):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk size")

    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk: 
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

Here, chunk_size=800 means approximately 800 characters, not tokens. overlap=150 means adjacent chunks share 150 characters to help preserve context across boundaries.

In [ ]:
# Sample
sample_chunks = chunk_text(pages[0]["text"])

print("Number of chunks:", len(sample_chunks))
print(sample_chunks[0])

Number of chunks: 4
9 UNIT 1 INTRODUCTION TO ARTIFICIAL INTELLIGENCE Structure 1.1 Introduction 1.2 Objectives 1.3 Basics of Artificial Intelligence (AI)? 1.4 Brief history of Artificial Intelligence 1.5 Components of Intelligence 1.6 Approaches to Artificial Intelligence 1.7 Comparison between Artificial Intelligence (AI), Machine Learning (ML) and DeepLearning (DL). 1.8 Application Areas of Artificial Intelligence Systems 1.9 Intelligent Agents 1.9.1 Stimulus - Response Agents 1.10 Summary 1.11 Solutions/Answers 1.12 Further Readings 1.1 INTRODUCTION Today, artificial intelligence is used in a wide variety of applications, including engineering, technology, the military, opinion mining, sentiment analysis, and many more. It is also used in more advanced domains, such as language processing and applications


# Step 6 : Create chunks for every page

In [29]:
document_chunks = []

for page in pages : 
    chunks = chunk_text(page["text"])

    for chunk_number, chunk in enumerate(chunks, start = 1) :
        document_chunks.append({
            "page" : page["page"],
            "chunk" : chunk_number,
            "text" : chunk
        })
print("Total Chunks", len(document_chunks))

Total Chunks 137


In [30]:
print(document_chunks[0])

{'page': 1, 'chunk': 1, 'text': '9 UNIT 1 INTRODUCTION TO ARTIFICIAL INTELLIGENCE Structure 1.1 Introduction 1.2 Objectives 1.3 Basics of Artificial Intelligence (AI)? 1.4 Brief history of Artificial Intelligence 1.5 Components of Intelligence 1.6 Approaches to Artificial Intelligence 1.7 Comparison between Artificial Intelligence (AI), Machine Learning (ML) and DeepLearning (DL). 1.8 Application Areas of Artificial Intelligence Systems 1.9 Intelligent Agents 1.9.1 Stimulus - Response Agents 1.10 Summary 1.11 Solutions/Answers 1.12 Further Readings 1.1 INTRODUCTION Today, artificial intelligence is used in a wide variety of applications, including engineering, technology, the military, opinion mining, sentiment analysis, and many more. It is also used in more advanced domains, such as language processing and applications'}


# Step 7 : Generate Embeddings

In [32]:
chunk_texts = [item["text"] for item in document_chunks]

if chunk_text:
    chunk_embeddings = model_small.encode(chunk_texts)
    print("Number of chunks",len(chunk_texts))
    print("Embedding shape", chunk_embeddings.shape)
else:
    print("No text chunks found")

Number of chunks 137
Embedding shape (137, 384)


In [37]:
import json 
import numpy as np 

if document_chunks:
    np.save("document_embeddings.npy",chunk_embeddings)

    with open("Chapter 41: document_chunks.json", "w", encoding="utf-8") as file:
        json.dump(
            document_chunks,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("Embeddings and chunk metadata saved.")

Embeddings and chunk metadata saved.


# Check the Json file with Chapter 41: Document chunks It has all the page number chunk and everything